# Exercise 5: PropBank argument detection

*Sequence Labelling · version 0.1.0*

> Use sense-annotated Turkish PropBank sentences (Şahin and Adalı, 2018; see the chapter’s references) and read the selected predicates’ role definitions. Measure the position of overt ARG0 spans relative to the predicate. For examples without an overt ARG0, inspect the annotation policy and context before treating the argument as omitted. What does agreement tell you, and what discourse evidence is needed to recover a referent? If you use TRopBank frames with a UD treebank instead, treat subjects as syntactic observations: `nsubj` is not a gold ARG0 label. Manually establish predicate senses and roles before making an SRL claim. Evaluate overt-argument labelling and implicit-argument recovery separately.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" "stanza==1.11.0" sklearn-crfsuite transformers sentencepiece requests` (the compiled `hfst` package needs Python 3.11 or 3.12; elsewhere install `pyhfst`, which the first cell uses instead). Stanza is pinned to 1.11.0 because the worked solutions use its Kazakh KazNERD checkpoint; models already on disk are used from `STANZA_RESOURCES_DIR` (default `~/stanza_resources`), and missing ones are downloaded. The datasets are read from commit-pinned GitHub files (KazNERD, KyrgyzNER, the Starlang Turkish NER data, TRopBank, the Turkish WordNet and UD 2.17 IMST), downloaded once into `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`) and checked against SHA-256 hashes. Taggers receive the **gold words**, so tokenisation errors do not enter the scores.

**Data.** The released TRopBank frame file (`StarlangSoftware/TurkishPropbank`) contains framesets but no annotated sentences. A frameset's id is the id of a verb sense in the Turkish WordNet (KeNet), which links it to a verb lemma. The sentences come from the UD 2.17 IMST training split. A verb qualifies only if **every** sense of its lemma defines ARG0, because the sense used in the sentence is not annotated. The analysis below counts syntactic subjects. Even if every frame defines ARG0, that does not establish that a particular `nsubj` fills that role. *Candidate antecedents* are nouns, proper nouns and pronouns in the previous two sentences of the same IMST document whose person and number agree with the verb (nouns count as third person, default singular).

In [ ]:
import collections, hashlib, os, random, re, statistics
from pathlib import Path
import requests

# Pinned public releases (commit-addressed raw files) and their SHA-256 hashes.
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_{}.txt"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/data/{}"
SHA256 = {
    KAZNERD.format("train"): "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KAZNERD.format("valid"): "29d584d1a0d9aca6d58b39eb8a93e154791d303ecff56ef93f12b3c889239bf1",
    KAZNERD.format("test"): "359a1f61f6a21280c6c47b4fde9a5adcd07411d62015aa6e32d940c51f163b7a",
    KYRGYZNER.format("KyrgyzNER_TRAIN.jsonl.txt"): "719a74a18cd99d6709c1e0592d0337aafade65aa97fe762e62f3f72a6b343a26",
    KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt"): "57a35a92139497819a2792d3e9e00a41b387b029b7851be7a64f4968befaddf5",
}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()

def fetch(url, sha256=None):
    """Download a pinned file once, check its hash and return the local path."""
    sha256 = sha256 or SHA256[url]
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != sha256:
        raise ValueError(f"{url} differs from the pinned release")
    return path

def read_conll(path, tag_col=-1, comment="#"):
    """Sentences as (words, tags) from a column file with blank-line sentence breaks."""
    sents, words, tags = [], [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words:
                sents.append((words, tags))
            words, tags = [], []
        elif not line.startswith(comment + " "):
            cols = line.split("\t") if "\t" in line else line.split()
            words.append(cols[0]); tags.append(cols[tag_col].strip())
    return sents

def spans(tags):
    """Strict IOB2 spans {(start, end, type)}; an I- tag that does not continue a span starts none."""
    out, start, typ = set(), None, None
    for i, t in enumerate(list(tags) + ["O"]):
        if t.startswith("I-") and typ == t[2:]:
            continue
        if typ is not None:
            out.add((start, i, typ))
        start, typ = (i, t[2:]) if t.startswith("B-") else (None, None)
    return out

def bioes_to_iob2(tags):
    return ["B-" + t[2:] if t[:2] in ("S-", "B-") else "I-" + t[2:] if t[:2] in ("E-", "I-") else "O" for t in tags]

def prf(gold, pred):
    """Micro precision, recall and F1 (%) over lists of span sets."""
    g = sum(len(x) for x in gold); p = sum(len(x) for x in pred)
    c = sum(len(x & y) for x, y in zip(gold, pred))
    pr = c / p if p else 0.0; rc = c / g if g else 0.0
    return {"gold": g, "pred": p, "correct": c, "P": round(100 * pr, 2), "R": round(100 * rc, 2),
            "F1": round(100 * (2 * pr * rc / (pr + rc) if pr + rc else 0.0), 2)}

def stanza_ner(lang="kk"):
    """Stanza NER over gold words; models from STANZA_RESOURCES_DIR, downloaded only if missing."""
    import logging, stanza
    logging.getLogger("stanza").setLevel(logging.ERROR)
    model_dir = os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser()))
    have = (Path(model_dir) / lang / "ner").exists()
    return stanza.Pipeline(lang, processors="tokenize,ner", tokenize_pretokenized=True, model_dir=model_dir,
                           download_method=None if have else "reuse_resources", verbose=False)

def tag_words(nlp, sents, batch=2000):
    """IOB2 predictions for gold-word sentences (spaces inside a token are joined with '_')."""
    out = []
    for i in range(0, len(sents), batch):
        doc = nlp([[w.replace(" ", "_") for w in words] for words, _ in sents[i:i + batch]])
        for s, (words, _) in zip(doc.sentences, sents[i:i + batch]):
            assert len(s.tokens) == len(words)
            out.append(bioes_to_iob2([t.ner for t in s.tokens]))
    return out

In [ ]:
import xml.etree.ElementTree as ET
TROPBANK = ("https://raw.githubusercontent.com/StarlangSoftware/TurkishPropbank/"
            "4578d1476b2eb62b8e2b07082152002d8ce07981/src/main/resources/turkish-propbank.xml",
            "c0493af82bdfe3511cce49972a4413db85fb8c79f0459569a421fd5c332eaf03")
WORDNET = ("https://raw.githubusercontent.com/StarlangSoftware/TurkishWordNet/"
           "89fee78195cac782010655cf1d04bae800c83fa5/src/main/resources/turkish_wordnet.xml",
           "afe256845a8e8d26ab097eb8efda9627aae61c8018cba69b566adc19699593f5")
IMST = ("https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-IMST/r2.17/tr_imst-ud-train.conllu",
        "8d8b8b581dab9362e6f2e4328aef77b9fabec1d189a199e320ba136710c7bd83")

frames = {fs.get("id"): {a.get("name"): (a.text or "").strip() for a in fs.findall("ARG")}
          for fs in ET.parse(fetch(*TROPBANK)).getroot().findall("FRAMESET")}
print(f"TRopBank: {len(frames)} framesets; with ARG0 {sum('ARG0' in f for f in frames.values())}; "
      f"with ARG2 {sum('ARG2' in f for f in frames.values())}")
arg2 = collections.Counter(f["ARG2"].split(",")[0].split()[-1] for f in frames.values() if f.get("ARG2"))
print("ARG2 is predicate-specific; most frequent last words of its descriptions:", arg2.most_common(8))

# Link frames to verbs through the Turkish WordNet (frameset id = synset id).
verb_frames = collections.defaultdict(list)
for syn in re.finditer(r"<SYNSET><ID>(TUR10-\d+)</ID>(.*?)</SYNSET>", fetch(*WORDNET).read_text(encoding="utf-8")):
    if syn.group(1) in frames and "<POS>v</POS>" in syn.group(2):
        for lit in re.findall(r"<LITERAL>([^<]+)<SENSE>", syn.group(2)):
            verb_frames[lit.strip()].append(frames[syn.group(1)])
print(f"{len(verb_frames)} verb lemmas have at least one frameset")

def imst_docs(path):
    """IMST training sentences as (doc id, words with id/form/lemma/upos/feats/head/deprel)."""
    out = []
    for block in Path(path).read_text(encoding="utf-8").strip().split("\n\n"):
        sid = re.search(r"# sent_id = (\S+)", block).group(1)
        ws = [c.split("\t") for c in block.splitlines() if not c.startswith("#")]
        ws = [dict(id=int(c[0]), form=c[1], lemma=c[2], upos=c[3], feats=c[5], head=int(c[6]), deprel=c[7])
              for c in ws if c[0].isdigit()]
        out.append((sid.rsplit("_", 1)[0], ws))
    return out

def infinitive(lemma):
    back = next((ch for ch in reversed(lemma) if ch in "aeıioöuü"), "a") in "aıou"
    return lemma + ("mak" if back else "mek")

docs = imst_docs(fetch(*IMST))
pos = collections.Counter(); implicit = []
for k, (doc, ws) in enumerate(docs):
    for w in ws:
        f = dict(x.split("=") for x in w["feats"].split("|") if "=" in x)
        fr = verb_frames.get(infinitive(w["lemma"]))
        if w["upos"] != "VERB" or not fr or "Pass" in f.get("Voice", "") or f.get("Mood") == "Imp":
            continue
        if not all("ARG0" in x for x in fr):          # every sense of the verb must define an ARG0
            continue
        subj = [d for d in ws if d["head"] == w["id"] and d["deprel"].split(":")[0] == "nsubj"]
        if subj:
            pos["before verb" if subj[0]["id"] < w["id"] else "after verb"] += 1
        elif "Person" in f and (f.get("VerbForm") in (None, "Fin")):
            # candidate antecedents: nominals in the previous two sentences of the same document that
            # agree in person and number (3rd person: nouns/proper nouns/3rd-person pronouns)
            prev = [d for j in (k - 2, k - 1) if j >= 0 and docs[j][0] == doc for d in docs[j][1]]
            def agrees(d):
                g = dict(x.split("=") for x in d["feats"].split("|") if "=" in x)
                person = g.get("Person", "3") if d["upos"] == "PRON" else "3"
                return (d["upos"] in ("NOUN", "PROPN", "PRON") and person == f["Person"]
                        and g.get("Number", "Sing") == f.get("Number", "Sing"))
            implicit.append((f["Person"], sum(agrees(d) for d in prev)))
n = sum(pos.values())
print(f"overt nsubj (standing in for ARG0) of active predicates whose frames all define ARG0: n={n}; "
      + ", ".join(f"{k} {100 * v / n:.1f}%" for k, v in pos.items()))
print(f"finite predicates with no overt subject: {len(implicit)}")
for person in ("1", "2", "3"):
    c = [x for p, x in implicit if p == person]
    if c:
        print(f"   Person={person}: n={len(c)}, agreeing candidates in the previous two sentences: "
              f"median {statistics.median(c):g}, none {100 * sum(x == 0 for x in c) / len(c):.0f}%, "
              f"exactly one {100 * sum(x == 1 for x in c) / len(c):.0f}%")

## Worked solution

Output of `experiments/ch09_exercises/run_ch09_exercises.py`:

```
TRopBank: 17531 framesets; with ARG0 14535; with ARG2 1865
ARG2 is predicate-specific; most frequent last words of its descriptions: [('kişi', 618), ('yer', 315), ('şey', 291), ('konu', 115), ('alet', 49), ('araç', 48), ('zaman', 43), ('alan', 35)]
18169 verb lemmas have at least one frameset
overt nsubj (standing in for ARG0) of active predicates whose frames all define ARG0: n=703; before verb 97.9%, after verb 2.1%
finite predicates with no overt subject: 1033
   Person=1: n=397, agreeing candidates in the previous two sentences: median 0, none 72%, exactly one 24%
   Person=2: n=96, agreeing candidates in the previous two sentences: median 0, none 75%, exactly one 18%
   Person=3: n=540, agreeing candidates in the previous two sentences: median 4, none 12%, exactly one 8%
```

**The frames.** 14,535 of the 17,531 framesets define an ARG0, and 1,865 define an ARG2. ARG2 is predicate-specific, as the chapter says: its descriptions most often end in *kişi* 'person', *yer* 'place', *şey* 'thing', *konu* 'topic', *alet* 'tool' or *zaman* 'time', which correspond to different roles in different frames.

**Overt subjects.** For 703 active predicates whose frames all define ARG0 and that have an overt `nsubj`, the subject precedes the verb in 97.9% of cases and follows it in 2.1%. Position is a strong cue in this edited text, but not a rule.

**Predicates without an overt subject.** 1,033 finite predicates have no overt subject. First- and second-person verbs (397 and 96) usually have *no* agreeing candidate in the previous two sentences (72% and 75%): agreement indicates a speaker or addressee role, but the code does not identify the person referred to. Third-person verbs (540) are the opposite: the median number of agreeing nominals is 4, only 8% have exactly one, and 12% have none. These are counts of candidates generated by a heuristic. They do not show which candidate is correct, whether a referent is recoverable, or why no candidate was found. Those questions require checked discourse and role annotations.

This worked example reports subject positions and candidate-set sizes. It does not complete the requested SRL evaluation: neither gold semantic roles nor gold antecedents are available here.

## Your turn — extension

1. Restrict the candidates to nominals whose head noun is a person according to the Turkish WordNet, and recompute the share with exactly one candidate.
2. Take the previous sentence's subject as the antecedent and check 50 cases by hand.